# Introdução ao Apache Spark

## O que é Apache Spark?

Apache Spark é um framework de processamento distribuído de código aberto projetado para computação rápida em grandes conjuntos de dados. Diferente do Hadoop MapReduce tradicional, o Spark processa dados em memória, o que o torna significativamente mais rápido para muitas aplicações.

### Principais características do Spark:

1. **Velocidade**: Execução em memória que pode ser até 100x mais rápida que o Hadoop para determinadas cargas de trabalho
2. **Facilidade de uso**: APIs em Python, Java, Scala e R
3. **Generalidade**: Suporta SQL, streaming, machine learning e processamento de grafos
4. **Execução em qualquer lugar**: Pode ser executado em Hadoop, Mesos, Kubernetes, standalone ou na nuvem

## Arquitetura do Spark

O Spark é baseado em dois conceitos principais:

1. **RDD (Resilient Distributed Dataset)**: Coleção distribuída de elementos que pode ser processada em paralelo
2. **DataFrame/Dataset**: APIs estruturadas de alto nível construídas sobre RDDs

A arquitetura do Spark inclui:
Componentes principais:

#### Driver Program:

Contém a aplicação principal e cria o SparkContext.
Responsável por converter o código do usuário em tarefas que serão executadas pelos executores.
Orquestra o trabalho distribuído e gerencia o estado da aplicação.


#### SparkContext:

Ponto de entrada para qualquer funcionalidade Spark.
Representa a conexão com o cluster Spark.
Responsável por configurar serviços internos e estabelecer conexão com o gerenciador de cluster.
Em PySpark moderno, muitas vezes substituído pelo objeto SparkSession que encapsula SparkContext.


#### Cluster Manager:

Gerencia recursos no cluster (YARN, Mesos, Kubernetes ou gerenciador standalone do Spark).
Aloca recursos e monitora a execução dos aplicativos Spark.


#### Executors:

Processos JVM iniciados em nós de trabalho.
Executam as tarefas atribuídas pelo driver.
Armazenam dados em cache quando solicitado.
Reportam resultados de volta ao driver.

### Vantagens do PySpark (combinação de Python e Spark)
###  Por que PySpark é valioso:

#### Combina a facilidade e popularidade do Python com o poder de processamento distribuído do Spark.
#### Permite aos cientistas de dados e engenheiros usar bibliotecas Python populares (NumPy, Pandas, scikit-learn) em conjunto com Spark.
#### Fornece APIs intuitivas para manipulação de dados complexos.
#### Integração com visualização de dados e notebooks (Jupyter, Zeppelin).
#### Suporte a processamento de streaming, machine learning (MLlib) e análise de grafos (GraphX).
#### Possibilita a transição mais suave para profissionais que já conhecem Python.

## Iniciando o SparkSession

O SparkSession é o ponto de entrada para programação Spark com a API Dataset e DataFrame. Vamos criar uma sessão Spark:

In [11]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Configuração otimizada
spark = SparkSession.builder \
    .appName("Spark Avançado") \
    .config("spark.network.timeout", "800s") \
    .config("spark.executor.heartbeatInterval", "400s") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "10").getOrCreate()

# Verificando a versão do Spark
print(f"Versão do Spark: {spark.version}")

Versão do Spark: 3.5.5


## Carregando Dados no Spark

Vamos carregar o arquivo CSV de vendas para começar a explorar o Spark:

In [12]:
# Caminho para o arquivo CSV
file_path = "sales.csv"

# Carregando o arquivo CSV como um DataFrame
df = spark.read.csv(file_path, header=True, inferSchema=True)

# Exibindo o schema do DataFrame
print("Schema do DataFrame:")
df.printSchema()


Schema do DataFrame:
root
 |-- Data: date (nullable = true)
 |-- ID_Cliente: string (nullable = true)
 |-- Nome_Cliente: string (nullable = true)
 |-- ID_Vendedor: string (nullable = true)
 |-- Nome_Vendedor: string (nullable = true)
 |-- Cidade: string (nullable = true)
 |-- Estado: string (nullable = true)
 |-- Categoria_Produto: string (nullable = true)
 |-- Produto: string (nullable = true)
 |-- Quantidade: integer (nullable = true)
 |-- Preco_Unitario: integer (nullable = true)
 |-- Desconto: double (nullable = true)
 |-- Valor_Total: double (nullable = true)
 |-- Custo: integer (nullable = true)
 |-- Lucro: double (nullable = true)
 |-- Metodo_Envio: string (nullable = true)
 |-- Frequencia_Compra: integer (nullable = true)



In [13]:

# Exibindo as primeiras linhas do DataFrame
print("\nPrimeiras 5 linhas do DataFrame:")
df.show(5)


Primeiras 5 linhas do DataFrame:
+----------+----------+---------------+-----------+-------------+--------------+------+-----------------+------------+----------+--------------+--------+-----------+-----+-----+------------+-----------------+
|      Data|ID_Cliente|   Nome_Cliente|ID_Vendedor|Nome_Vendedor|        Cidade|Estado|Categoria_Produto|     Produto|Quantidade|Preco_Unitario|Desconto|Valor_Total|Custo|Lucro|Metodo_Envio|Frequencia_Compra|
+----------+----------+---------------+-----------+-------------+--------------+------+-----------------+------------+----------+--------------+--------+-----------+-----+-----+------------+-----------------+
|2023-01-05|      C001|     João Silva|       V001| Ana Oliveira|     São Paulo|    SP|      Eletrônicos|  Smartphone|         2|          1200|    0.05|     2280.0| 1800|480.0|    Expresso|                3|
|2023-01-10|      C002|   Maria Santos|       V002|  Bruno Costa|Rio de Janeiro|    RJ|           Móveis|        Sofá|         1| 

##  Usando withColumn para criar colunas derivadas


In [14]:
df_transformado = df.withColumn("faixa_preco", 
                               F.when(F.col("Valor_Total") < 3000, "Baixo")
                                .when(F.col("Valor_Total") < 4000, "Médio")
                                .otherwise("Alto"))

df_transformado.show(5)                                

+----------+----------+---------------+-----------+-------------+--------------+------+-----------------+------------+----------+--------------+--------+-----------+-----+-----+------------+-----------------+-----------+
|      Data|ID_Cliente|   Nome_Cliente|ID_Vendedor|Nome_Vendedor|        Cidade|Estado|Categoria_Produto|     Produto|Quantidade|Preco_Unitario|Desconto|Valor_Total|Custo|Lucro|Metodo_Envio|Frequencia_Compra|faixa_preco|
+----------+----------+---------------+-----------+-------------+--------------+------+-----------------+------------+----------+--------------+--------+-----------+-----+-----+------------+-----------------+-----------+
|2023-01-05|      C001|     João Silva|       V001| Ana Oliveira|     São Paulo|    SP|      Eletrônicos|  Smartphone|         2|          1200|    0.05|     2280.0| 1800|480.0|    Expresso|                3|      Baixo|
|2023-01-10|      C002|   Maria Santos|       V002|  Bruno Costa|Rio de Janeiro|    RJ|           Móveis|        Sof

### 3. Agregações mais avançadas

In [15]:

resumo_PorCidade = df.groupBy("Cidade") \
                .agg(
                    F.count("ID_Cliente").alias("Quantidade_Clientes"),
                    F.round(F.avg("Valor_Total"), 2).alias("Media de Vendas"),
                    F.min("Valor_Total").alias("Menor Venda"),
                    F.max("Valor_Total").alias("Maior Venda"),
                    F.round(F.stddev("Valor_Total"), 2).alias("Desvio Padrao de Vendas")
                )

resumo_PorCidade.show()                

+--------------+-------------------+---------------+-----------+-----------+-----------------------+
|        Cidade|Quantidade_Clientes|Media de Vendas|Menor Venda|Maior Venda|Desvio Padrao de Vendas|
+--------------+-------------------+---------------+-----------+-----------+-----------------------+
|Rio de Janeiro|                  8|        1805.63|      650.0|     3420.0|                 916.31|
|  Porto Alegre|                  4|          481.5|       70.0|      950.0|                  380.2|
|Belo Horizonte|                  7|        1557.71|      114.0|     3500.0|                1415.13|
|      Curitiba|                  4|          458.0|      300.0|      850.0|                 262.05|
|     São Paulo|                  8|        1190.44|      351.0|     2280.0|                 747.74|
|      Salvador|                  4|          206.9|       80.0|      360.0|                 118.64|
|     Fortaleza|                  3|         677.67|      266.0|     1425.0|               

### Joins Avançados

In [17]:
# Criando um segundo DataFrame para joins
dados_projetos = [
    ("projeto1", "CRM", "2023-01-10", "2023-05-15"),
    ("projeto2", "ERP", "2022-08-20", "2023-02-28"),
    ("projeto3", "App Mobile", "2023-03-05", "2023-07-20"),
    ("projeto4", "Dashboard", "2023-02-15", "2023-04-30"),
    ("projeto5", "API", "2023-04-10", None)
]

schema_projetos = ["id_projeto", "nome_projeto", "data_inicio", "data_fim"]
df_projetos = spark.createDataFrame(dados_projetos, schema_projetos)

# Inner join

In [18]:
dados_responsaveis = [
    ("projeto1", "Carlos"),
    ("projeto2", "Maria"),
    ("projeto3", "Ana"),
    ("projeto6", "João")  # Este projeto não existe no df_projetos
]

schema_responsaveis = ["id_projeto", "responsavel"]

df_responsaveis = spark.createDataFrame(dados_responsaveis, schema_responsaveis)

## # Executando o join com cache para melhorar performance

In [19]:
df_projetos.cache()
df_responsaveis.cache()

DataFrame[id_projeto: string, responsavel: string]

### Forçando a execução do cache


In [ ]:
df_projetos.count()
df_responsaveis.count()

###  Inner Join (apenas correspondências)

In [ ]:
df_projetos.join(df_responsaveis, on="id_projeto", how="inner").show()


 ### Left Join (todos os projetos, mesmo sem responsável)

In [ ]:
df_projetos.join(df_responsaveis, on="id_projeto", how="left").show()

### Right Join (todos os responsáveis, mesmo sem projeto)

In [ ]:
df_projetos.join(df_responsaveis, on="id_projeto", how="right").show()


### Verificando Registros Não Correspondentes (anti-join)

In [ ]:
# Projetos sem responsável
df_projetos.join(df_responsaveis, on="id_projeto", how="left_anti").show()


## Window Functions
[]: #
[]: # As Window Functions permitem realizar cálculos em uma janela de dados, como a soma acumulada de uma coluna.
[]: #
[]: # ### 1. Usando `sum` em uma janela de dados
[]: #
[]: # Vamos usar a função `sum` em uma janela de dados para calcular a soma acumulada de uma coluna.
[]: #

In [ ]:
# Criando um DataFrame de exemplo com dados de funcionários
dados_funcionarios = [
    ("James", "Vendas", 3000),
    ("Michael", "Vendas", 4600),
    ("Robert", "Vendas", 4100),
    ("Maria", "Finanças", 3000),
    ("James", "Finanças", 3000),
    ("Scott", "Finanças", 3300),
    ("Jen", "Finanças", 3900),
    ("Jeff", "Marketing", 3000),
    ("Kumar", "Marketing", 2000)
]
colunas = ["nome_funcionario", "departamento", "salario"]
df_funcionarios = spark.createDataFrame(data=dados_funcionarios, schema=colunas)

print("DataFrame Original:")
df_funcionarios.show()

In [ ]:
janela_departamento = Window.partitionBy("departamento")

### Adicionando uma nova coluna 'rank_salario_depto'
### A função F.rank() atribui um rank a cada funcionário baseado no salário dentro do seu departamento

In [ ]:
df_com_ranking = df_com_media_salario.withColumn( # Podemos usar o df anterior
    "rank_salario_depto",
    F.rank().over(janela_depto_ordenada_salario)
)

print("DataFrame com Ranking de Salário por Departamento:")
df_com_ranking.show()

NameError: name 'df_com_media_salario' is not defined

## Encerrando a Sessão Spark

Quando terminarmos de usar o Spark, devemos encerrar a sessão para liberar recursos:

In [ ]:
# Encerrando a sessão Spark
spark.stop()